# Employee Exit Survey Analytics

Exploratory analysis of employee exit surveys from **DETE** and **TAFE**.  
Pipeline: **Load → Clean → Transform → Validate → Combine → Analyze → Visualize**.

All logic lives in `src/` and is unit-tested with `pytest`. Run the full pipeline with:
```bash
python scripts/run_pipeline.py
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

import src.load as ld
import src.clean as cl
import src.transform as tf
import src.combine as cb
import src.validate as va
import src.analyze as az
import src.visualize as vz

## 1. Load & profile the raw surveys

In [ ]:
raw_dete = ld.load_dete()
raw_tafe = ld.load_tafe()
print('DETE raw shape:', raw_dete.shape)
print('TAFE raw shape:', raw_tafe.shape)
ld.profile(raw_dete).head(12)

## 2. Clean
Standardize headers, drop irrelevant columns, normalize missing markers, remove duplicates.

In [ ]:
dete_clean, dete_rep = cl.clean_dete(raw_dete)
tafe_clean, tafe_rep = cl.clean_tafe(raw_tafe)
print('DETE dropped columns:', dete_rep['dropped_columns'])
print('DETE duplicates removed:', dete_rep['duplicates_dropped'])
print('TAFE duplicates removed:', tafe_rep['duplicates_dropped'])
dete_clean.head()

## 3. Transform
Parse dates → tenure, map age ranges, build age/tenure groups, apply the dissatisfaction business rule.

In [ ]:
dete_tx = tf.transform_dete(dete_clean)
tafe_tx = tf.transform_tafe(tafe_clean)
dete_tx[['id','separation_type','cease_year','age','length_of_service','dissatisfied','contributing_factors']].head()

## 4. Validate & Combine

In [ ]:
combined = cb.combine_datasets(dete_tx, tafe_tx)
report = va.validate(combined)
print(report)
combined.head()

## 5. Analyze

In [ ]:
print(az.summarize(combined))
az.compare_institutes(combined)

In [ ]:
az.top_resignation_reasons(combined)

In [ ]:
az.dissatisfaction_by_age(combined)

In [ ]:
az.significant_patterns(combined)

## 6. Visualize

In [ ]:
paths = vz.generate_all_figures(combined)
for p in paths: print(p)

In [ ]:
from IPython.display import Image, display
for p in paths:
    display(Image(filename=p))